In [14]:
"""
Split netCDF data into datasets to plot vertical profile and stability indices
"""

from pathlib import Path
import xarray as xr
import numpy as np

cycles = ["00","06","12","18"]

# BASE_PATH = Path.cwd().resolve().parent.parent.parent / "sif" / "data" / "IFS"
# path = "2026-08-13/18z/ifs/20260813180000-12h-oper-fc.nc"

# for i in cycles:
#     input_folder = BASE_PATH / date_input / f"{i}z" / "ifs" 
#     output_folder = BASE_PATH / date_input / "netCDF" / f"{i}z" 
#     output_folder.mkdir(parents=True, exist_ok=True)

# load data
ds = xr.open_dataset(BASE_PATH / path)
 

# vertical profile


In [2]:
ds

<xarray.Dataset> Size: 743MB
Dimensions:             (latitude: 721, longitude: 1440, soilLayer: 4,
                         isobaricInhPa: 14)
Coordinates: (12/13)
  * latitude            (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude           (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
  * soilLayer           (soilLayer) float64 32B 1.0 2.0 3.0 4.0
  * isobaricInhPa       (isobaricInhPa) float64 112B 1e+03 925.0 ... 50.0 10.0
    time                datetime64[ns] 8B ...
    step                timedelta64[ns] 8B ...
    ...                  ...
    valid_time          datetime64[ns] 8B ...
    heightAboveGround   float64 8B ...
    entireAtmosphere    float64 8B ...
    mostUnstableParcel  float64 8B ...
    nominalTop          float64 8B ...
    meanSea             float64 8B ...
Data variables: (12/43)
    tp                  (latitude, longitude) float32 4MB ...
    sp                  (latitude, longitude) float32 4MB ...
    sve                 (latitude, longitude) float32 4MB ...
    mx2t3               (latitude, longitude) float32 4MB ...
    tcw                 (latitude, longitude) float32 4MB ...
    svn                 (latitude, longitude) float32 4MB ...
    ...                  ...
    u                   (isobaricInhPa, latitude, longitude) float32 58MB ...
    v                   (isobaricInhPa, latitude, longitude) float32 58MB ...
    ro                  (latitude, longitude) float32 4MB ...
    sd                  (latitude, longitude) float32 4MB ...
    sf                  (latitude, longitude) float32 4MB ...
    msl                 (latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-14T14:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
# select 4 stations
# variables: t, r, u, v
vars = [
    "t", # temperature
    "r", # relative humidity
    "u", # u component of wind
    "v" # v component of wind
]

# stations
stations = [
    (54.527846, 11.060437), # Fehmarn
    (54.528, 9.55), # Schleswig
    (54.097, 13.405), # Greifswald
    (53.712, 7.152) # Norderney
]

In [4]:
# Select each point
selected = []

for lat, lon in stations:
    tmp = ds[vars].sel(
        latitude=lat,
        longitude=lon,
        method="nearest"
    )
    selected.append(tmp)

# Combine points into a new dimension
new_ds = xr.concat(selected, dim="point")

# Add point number and coordinates
new_ds = new_ds.assign_coords(
    point=np.arange(1, len(stations) + 1),
    point_lat=("point", [p[0] for p in stations]),
    point_lon=("point", [p[1] for p in stations])
)

# Save
# new_ds.to_netcdf("new.nc")


In [5]:
ds2 = xr.open_dataset("new.nc")

In [6]:
ds2

<xarray.Dataset> Size: 1kB
Dimensions:             (point: 4, isobaricInhPa: 14)
Coordinates: (12/15)
  * point               (point) int64 32B 1 2 3 4
    latitude            (point) float64 32B ...
    longitude           (point) float64 32B ...
    point_lat           (point) float64 32B ...
    point_lon           (point) float64 32B ...
  * isobaricInhPa       (isobaricInhPa) float64 112B 1e+03 925.0 ... 50.0 10.0
    ...                  ...
    valid_time          datetime64[ns] 8B ...
    heightAboveGround   float64 8B ...
    entireAtmosphere    float64 8B ...
    mostUnstableParcel  float64 8B ...
    nominalTop          float64 8B ...
    meanSea             float64 8B ...
Data variables:
    t                   (point, isobaricInhPa) float32 224B ...
    r                   (point, isobaricInhPa) float32 224B ...
    u                   (point, isobaricInhPa) float32 224B ...
    v                   (point, isobaricInhPa) float32 224B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-14T14:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [13]:
levels = [1000,925,850]
ds2.sel(point = 1)["r"]#.coords["time"]

<xarray.DataArray 'r' (isobaricInhPa: 14)> Size: 56B
[14 values with dtype=float32]
Coordinates: (12/15)
  * isobaricInhPa       (isobaricInhPa) float64 112B 1e+03 925.0 ... 50.0 10.0
    time                datetime64[ns] 8B ...
    step                timedelta64[ns] 8B ...
    surface             float64 8B ...
    latitude            float64 8B ...
    longitude           float64 8B ...
    ...                  ...
    mostUnstableParcel  float64 8B ...
    nominalTop          float64 8B ...
    meanSea             float64 8B ...
    point               int64 8B 1
    point_lat           float64 8B ...
    point_lon           float64 8B ...
Attributes: (12/30)
    GRIB_paramId:                             157
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                Relative humidity
    GRIB_shortName:                           r
    GRIB_units:                               %
    long_name:                                Relative humidity
    units:                                    %
    standard_name:                            relative_humidity

### Random

In [19]:
ds = xr.open_dataset("/Users/appa/Desktop/Masters/Semester2/Experimental_Meterology/sif/data/IFS/2026-08-15/netCDF/00z/20260815000000-12h-oper-fc.nc")

In [22]:
ds.dims

FrozenMappingWarningOnValuesAccess({'station': 4, 'soilLayer': 4, 'isobaricInhPa': 14})